In [1]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import os
import matplotlib.pylab as plt
from sklearn.metrics import roc_curve, roc_auc_score, recall_score

In [2]:
train_dir = r'/home/kchen/Documents/iom/data/split/train'
val_dir = r'/home/kchen/Documents/iom/data/split/val'
data_dir = r'/home/kchen/Documents/iom/data/jpg'
test_dir = r'/home/kchen/Documents/iom/data/split/test'

In [3]:
batch_size = 1
img_height = 512
img_width = 512

In [4]:
#load images into datasets with 'train' set split into training and validation
train_ds = tf.keras.preprocessing.image_dataset_from_directory(train_dir, label_mode='binary', subset='training', validation_split = 0.15, seed=0, image_size=(img_height, img_width), batch_size=batch_size, color_mode='rgb')
val_ds = tf.keras.preprocessing.image_dataset_from_directory(train_dir, label_mode='binary', subset='validation', validation_split = 0.15, seed=0, image_size=(img_height, img_width), batch_size=batch_size, color_mode='rgb')


Found 359 files belonging to 2 classes.
Using 306 files for training.
Found 359 files belonging to 2 classes.
Using 53 files for validation.


In [5]:
#normalize and apply data augmentation
normalization_layer = tf.keras.layers.experimental.preprocessing.Rescaling(1. / 255)
preprocessing_model = tf.keras.Sequential([normalization_layer])
do_data_augmentation = True
if do_data_augmentation:
  preprocessing_model.add(
      tf.keras.layers.experimental.preprocessing.RandomRotation(40))
  preprocessing_model.add(
      tf.keras.layers.experimental.preprocessing.RandomTranslation(0.2, 0.2))
  preprocessing_model.add(
      tf.keras.layers.experimental.preprocessing.RandomZoom(0.2, 0.2))
  preprocessing_model.add(
      tf.keras.layers.experimental.preprocessing.RandomFlip(mode="horizontal"))
  preprocessing_model.add(
      tf.keras.layers.experimental.preprocessing.RandomFlip(mode="vertical"))

train_ds = train_ds.map(lambda images, labels:
                        (preprocessing_model(images), labels))


In [6]:
#only apply normalization to validation set
val_ds = val_ds.unbatch().batch(batch_size)
val_ds = val_ds.map(lambda images, labels:
                    (normalization_layer(images), labels))

In [7]:
#download Xception base model
base_model = keras.applications.xception.Xception(weights='imagenet', input_shape=(img_height, img_height, 3), include_top=False)
base_model.trainable = False

In [8]:
#add input layer and outputs, last layer is binary classifier
inputs = keras.Input(shape=(img_height, img_height, 3))
x = base_model(inputs, training=False)
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.Dropout(0.4)(x)
outputs = keras.layers.Dense(1, activation='sigmoid')(x)
model = keras.Model(inputs, outputs)

In [9]:
#compile and train
model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
              loss=keras.losses.BinaryCrossentropy(from_logits=False),
              metrics=[keras.metrics.BinaryAccuracy(), keras.metrics.AUC(), keras.metrics.Recall(), keras.metrics.TrueNegatives(), keras.metrics.FalseNegatives(), keras.metrics.TruePositives(), keras.metrics.FalsePositives()])

early_stopping = keras.callbacks.EarlyStopping(
    patience=3,
    min_delta=0.001,
    restore_best_weights=True,)

model.fit(train_ds, epochs=20, validation_data=val_ds, callbacks=early_stopping)


Epoch 1/20
306/306 [==============================] - 66s 187ms/step - loss: 0.7191 - binary_accuracy: 0.4935 - auc: 0.4785 - recall: 0.2979 - true_negatives: 109.0000 - false_negatives: 99.0000 - true_positives: 42.0000 - false_positives: 56.0000 - val_loss: 0.7081 - val_binary_accuracy: 0.5283 - val_auc: 0.5829 - val_recall: 1.0000 - val_true_negatives: 3.0000 - val_false_negatives: 0.0000e+00 - val_true_positives: 25.0000 - val_false_positives: 25.0000
Epoch 2/20
306/306 [==============================] - 55s 179ms/step - loss: 0.6725 - binary_accuracy: 0.5686 - auc: 0.5924 - recall: 0.4681 - true_negatives: 108.0000 - false_negatives: 75.0000 - true_positives: 66.0000 - false_positives: 57.0000 - val_loss: 0.7367 - val_binary_accuracy: 0.5660 - val_auc: 0.5986 - val_recall: 0.9600 - val_true_negatives: 6.0000 - val_false_negatives: 1.0000 - val_true_positives: 24.0000 - val_false_positives: 22.0000
Epoch 3/20
306/306 [==============================] - 54s 176ms/step - loss: 0.6750 

In [10]:
#unfreeze and train at low learning rate
base_model.trainable = True

model.compile(
    optimizer=keras.optimizers.Adam(1e-5), 
    loss=keras.losses.BinaryCrossentropy(from_logits=False),
    metrics=[keras.metrics.BinaryAccuracy(), keras.metrics.AUC(), keras.metrics.Recall(), keras.metrics.TrueNegatives(), keras.metrics.FalseNegatives(), keras.metrics.TruePositives(), keras.metrics.FalsePositives()],
)

epochs = 10
model.fit(train_ds, epochs=epochs, validation_data=val_ds, callbacks=early_stopping)

Epoch 1/10
306/306 [==============================] - 77s 225ms/step - loss: 0.6567 - binary_accuracy: 0.6013 - auc_1: 0.6347 - recall_1: 0.4894 - true_negatives_1: 115.0000 - false_negatives_1: 72.0000 - true_positives_1: 69.0000 - false_positives_1: 50.0000 - val_loss: 0.6807 - val_binary_accuracy: 0.5283 - val_auc_1: 0.5986 - val_recall_1: 0.6000 - val_true_negatives_1: 13.0000 - val_false_negatives_1: 10.0000 - val_true_positives_1: 15.0000 - val_false_positives_1: 15.0000
Epoch 2/10
306/306 [==============================] - 66s 217ms/step - loss: 0.6575 - binary_accuracy: 0.6176 - auc_1: 0.6459 - recall_1: 0.5177 - true_negatives_1: 116.0000 - false_negatives_1: 68.0000 - true_positives_1: 73.0000 - false_positives_1: 49.0000 - val_loss: 0.6700 - val_binary_accuracy: 0.5849 - val_auc_1: 0.6193 - val_recall_1: 0.7600 - val_true_negatives_1: 12.0000 - val_false_negatives_1: 6.0000 - val_true_positives_1: 19.0000 - val_false_positives_1: 16.0000
Epoch 3/10
306/306 [=================

In [11]:
model.save('xcept.h5')

/home/kchen/.local/lib/python3.9/site-packages/keras/utils/generic_utils.py:494: CustomMaskWarning: Custom mask layers require a config and must override get_config. When loading, the custom mask layer must be passed to the custom_objects argument.
  warnings.warn('Custom mask layers require a config and must override '


In [12]:
#load test set and normalize using same code
test_ds = tf.keras.preprocessing.image_dataset_from_directory(test_dir, label_mode='binary', seed=0, image_size=(img_height, img_width), batch_size=batch_size, color_mode='rgb')
test_ds = test_ds.unbatch().batch(batch_size)
test_ds = test_ds.map(lambda images, labels:
                    (normalization_layer(images), labels))


Found 91 files belonging to 2 classes.


In [13]:
#evaluate on test set using evaluate
model.evaluate(test_ds)

91/91 [==============================] - 2s 26ms/step - loss: 0.5905 - binary_accuracy: 0.6813 - auc_1: 0.7891 - recall_1: 0.9286 - true_negatives_1: 23.0000 - false_negatives_1: 3.0000 - true_positives_1: 39.0000 - false_positives_1: 26.0000


[0.5904825925827026,
 0.6813187003135681,
 0.7891156673431396,
 0.9285714030265808,
 23.0,
 3.0,
 39.0,
 26.0]

In [14]:
#generate predictions on test set
preds = model.predict(test_ds)
preds.shape

(91, 1)

In [15]:
#get labels by iterating through test set and appending labels to empty array
test_labels = np.array([])
for images, labels in test_ds:
    test_labels = np.append(test_labels, labels[0].numpy(), axis=0)
test_labels.shape

(91,)

In [16]:
#seems like the labels and preds don't match up
roc_auc_score(test_labels, preds)

0.565597667638484